In [ ]:
import subprocess, sys, random, numpy as np, torch
SEED = 266
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
commit = subprocess.run(['git','rev-parse','--short','HEAD'], capture_output=True, text=True).stdout.strip()
print({'commit': commit, 'python': sys.version.split()[0], 'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu', 'seed': SEED})

# Part 1 — Multimodal RAG
This notebook is the Kaggle entry point. Copy the three PDFs, `kb_manifest.json`, `Questions.csv`, and the validator into `part1/data/` before running all cells. Set `ANSWER_MODEL_NAME` to a local instruction model when GPU memory permits; leave it empty for the deterministic fallback.

In [ ]:
from pathlib import Path
import os, sys, json
def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'part1' / 'src').exists(): return candidate
    return Path.cwd()
ROOT = find_project_root()
sys.path.insert(0, str(ROOT / 'part1' / 'src'))
DATA_DIR = Path(os.environ.get('DATA_DIR', ROOT / 'part1' / 'data'))
OUT_DIR = Path(os.environ.get('OUTPUT_DIR', ROOT / 'part1' / 'outputs'))
os.environ.setdefault('EMBEDDING_MODEL_NAME','BAAI/bge-small-en-v1.5')
os.environ.setdefault('ANSWER_MODEL_NAME','Qwen/Qwen2.5-3B-Instruct')
os.environ.setdefault('USE_LOCAL_LLM','1')
os.environ.setdefault('USE_VLM','1')
os.environ.setdefault('VISION_MODEL_NAME','Salesforce/blip-image-captioning-base')
print({'root': str(ROOT), 'data_dir': str(DATA_DIR), 'output_dir': str(OUT_DIR)})

In [ ]:
from run_pipeline import run
metrics = run(DATA_DIR, OUT_DIR)
print(json.dumps(metrics, indent=2))

In [ ]:
import pandas as pd
submission = pd.read_csv(OUT_DIR / 'submission.csv', dtype=str, keep_default_na=False)
required = ['id','conversation_id','question_id','answer','figure_references','citations','status']
assert list(submission.columns) == required
assert len(submission) == len(pd.read_csv(DATA_DIR / 'Questions.csv', dtype=str, keep_default_na=False))
assert submission['status'].isin(['answered','not_found','conflict','false_premise','session_override']).all()
print(submission.head(3).to_string(index=False))

## Golden-set evaluation
Create `part1/data/golden_set.json` with exactly 25 original questions before tuning. The next cell computes answer accuracy, Recall@5, citation accuracy, and (when supplied) human groundedness ratings from generated files.

In [ ]:
golden = DATA_DIR / 'golden_set.json'
if golden.exists():
    from rag import GroundedAnswerer, HybridRetriever, load_records
    from evaluate import evaluate, human_summary
    records = load_records(DATA_DIR, OUT_DIR / 'figures')
    retriever = HybridRetriever(records, model_name=os.environ.get('EMBEDDING_MODEL_NAME','BAAI/bge-small-en-v1.5'))
    evaluation = evaluate(golden, retriever, GroundedAnswerer(os.environ.get('ANSWER_MODEL_NAME') or None))
    (OUT_DIR / 'golden_metrics.json').write_text(json.dumps(evaluation, indent=2))
    print(json.dumps({k:v for k,v in evaluation.items() if k != 'answers'}, indent=2))
else:
    print('golden_set.json not found; no evaluation metric was fabricated.')
ratings = DATA_DIR / 'human_ratings.csv'
if ratings.exists(): print(human_summary(ratings))